# 🧠 Day 51 + 52: Gradient Descent — Complete Deep Dive
### CampusX 100 Days of Machine Learning — Enhanced with Deep Comments & Explanations

---

## 📌 What is Gradient Descent?

Gradient Descent is an **optimization algorithm** used to minimize a function (usually a **loss/cost function**) by iteratively moving in the direction of the **steepest descent** (i.e., opposite to the gradient).

### 🏔️ Real-World Analogy:
> Imagine you're blindfolded on a hilly terrain and you want to reach the lowest valley.
> You feel the slope under your feet and take a step in the direction that goes downhill.
> You keep doing this until you reach a flat area — that's the minimum!

### 🎯 Why Do We Need It?
In Machine Learning (especially Linear Regression, Neural Networks), we need to find the best parameters (weights) that **minimize the error/loss**. We can't always solve this analytically (using formulas), so we use Gradient Descent.

---

## 📐 The Core Formula

$$\theta_{new} = \theta_{old} - \alpha \cdot \frac{\partial J(\theta)}{\partial \theta}$$

Where:
- $\theta$ = parameter (weight/bias) we want to update
- $\alpha$ = **Learning Rate** (how big a step we take)
- $J(\theta)$ = **Cost/Loss Function** (what we're minimizing)
- $\frac{\partial J}{\partial \theta}$ = **Gradient** (slope of the cost function at current point)

---

## 🧩 Key Concepts Before Code

| Term | Meaning |
|------|--------|
| **Loss Function** | Measures how wrong our model is (e.g., MSE) |
| **Gradient** | Partial derivative — tells direction of steepest ASCENT |
| **Learning Rate (α)** | Step size — too big = overshoot, too small = slow |
| **Convergence** | When loss stops decreasing significantly |
| **Epoch** | One full pass through the training data |
| **Global Minimum** | The lowest point of the loss surface |
| **Local Minimum** | A dip that isn't the absolute lowest |


In [ ]:
# ============================================================
#  📦 IMPORTS
# ============================================================
# numpy   → for fast numerical computations and matrix operations
# pandas  → for data loading and manipulation
# matplotlib / seaborn → for plotting loss curves and data
# sklearn → for data splitting and comparison with built-in models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set a consistent plot style
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)  # For reproducibility — same results every run

print("✅ All imports successful!")

---
## 🔬 Part 1: Understanding Gradient Descent on a Simple Function

Before applying it to ML, let's understand GD on a **simple 1D function**.

We'll minimize: $f(x) = x^2$

- Its minimum is at $x = 0$, $f(0) = 0$
- Gradient (derivative): $f'(x) = 2x$
- Update rule: $x_{new} = x_{old} - \alpha \cdot 2x_{old}$

In [ ]:
# ============================================================
#  🎯 GRADIENT DESCENT ON f(x) = x²  (1D Example)
# ============================================================

def gradient_descent_1d(
    start_x,      # Where we begin on the curve
    learning_rate, # How big each step is (alpha)
    n_iterations  # How many steps to take
):
    """
    Demonstrates Gradient Descent on f(x) = x^2
    
    We know:
      - f(x)  = x^2          (the function)
      - f'(x) = 2x           (the gradient / derivative)
      - Minimum is at x=0    (where derivative = 0)
    """
    x = start_x           # Start at initial position
    history = [x]         # Track all x positions visited
    
    for i in range(n_iterations):
        gradient = 2 * x  # Derivative of x^2 is 2x
        
        # 🔑 CORE UPDATE RULE:
        # Move OPPOSITE to gradient (that's the downhill direction)
        # If gradient is positive (slope goes up), we move left (x decreases)
        # If gradient is negative (slope goes down), we move right (x increases)
        x = x - learning_rate * gradient
        
        history.append(x)
    
    return x, history

# Run with different learning rates to see the effect
x_range = np.linspace(-5, 5, 300)
y_range = x_range ** 2  # f(x) = x^2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
learning_rates = [0.1, 0.9, 1.1]  # Good, aggressive, too large
titles = ['Good LR (α=0.1)', 'Aggressive LR (α=0.9)', 'Too Large LR (α=1.1) — Diverges!']
colors = ['green', 'orange', 'red']

for ax, lr, title, color in zip(axes, learning_rates, titles, colors):
    final_x, history = gradient_descent_1d(start_x=4.0, learning_rate=lr, n_iterations=20)
    
    ax.plot(x_range, y_range, 'b-', linewidth=2, label='f(x) = x²')
    
    # Plot only first 15 steps for visibility
    hist_arr = np.array(history[:15])
    ax.scatter(hist_arr, hist_arr**2, c=color, s=60, zorder=5, label='Steps taken')
    ax.plot(hist_arr, hist_arr**2, c=color, alpha=0.5, linewidth=1.5)
    
    ax.scatter([0], [0], c='gold', s=200, marker='*', zorder=10, label='True Minimum')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend()
    ax.set_ylim(-1, 20)

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"With α=0.1, final x = {gradient_descent_1d(4.0, 0.1, 20)[0]:.6f}  (close to 0 ✅)")
print(f"With α=0.9, final x = {gradient_descent_1d(4.0, 0.9, 20)[0]:.6f}  (might oscillate)")
print(f"With α=1.1, final x = {gradient_descent_1d(4.0, 1.1, 20)[0]:.6f}  (DIVERGED! ❌)")

---
## 🏗️ Part 2: Building Linear Regression Loss Function

For Linear Regression: $\hat{y} = m \cdot x + b$

**Mean Squared Error (MSE) Loss:**
$$J(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n} \sum_{i=1}^{n} (y_i - m x_i - b)^2$$

**Gradients (Partial Derivatives):**
$$\frac{\partial J}{\partial m} = -\frac{2}{n} \sum_{i=1}^{n} x_i(y_i - \hat{y}_i)$$
$$\frac{\partial J}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$

In [ ]:
# ============================================================
#  📊 CREATE SYNTHETIC DATASET FOR LINEAR REGRESSION
# ============================================================

# Generate synthetic placement data:
# CGPA (x) → Package/Salary (y) with some noise

n_samples = 200

# True underlying relationship: y = 3.5 * x + 2 + noise
# (We'll try to RECOVER these values using Gradient Descent)
true_slope     = 3.5   # m
true_intercept = 2.0   # b

X = np.random.uniform(5, 10, n_samples)        # CGPA: between 5 and 10
noise = np.random.normal(0, 1.5, n_samples)    # Random noise (mean=0)
y = true_slope * X + true_intercept + noise    # Target: salary in LPA

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Quick visualization
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, alpha=0.6, label='Train data', color='steelblue')
plt.scatter(X_test, y_test, alpha=0.6, label='Test data', color='orange')
plt.xlabel('CGPA')
plt.ylabel('Package (LPA)')
plt.title('Synthetic Placement Dataset\n(True: y = 3.5x + 2 + noise)', fontsize=12)
plt.legend()
plt.show()

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"True slope = {true_slope}, True intercept = {true_intercept}")
print("Goal: Gradient Descent should discover these values!")

---
## ⚙️ Part 3: Batch Gradient Descent from Scratch

### What is Batch Gradient Descent?
- Uses the **ENTIRE dataset** to compute the gradient at each step
- Very stable, smooth convergence
- Slow for large datasets (processes all data before one update)
- Called "batch" because it processes the whole batch of data each time

```
for each epoch:
    gradient = compute using ALL training samples
    update parameters
```

In [ ]:
# ============================================================
#  🔵 BATCH GRADIENT DESCENT — Full Implementation from Scratch
# ============================================================

class BatchGradientDescent:
    """
    Implements Linear Regression using BATCH Gradient Descent.
    
    In each iteration (epoch), we:
    1. Compute predictions using CURRENT m and b
    2. Compute error = actual - predicted
    3. Compute gradient of MSE w.r.t. m and b (using ALL data)
    4. Update m and b by subtracting learning_rate * gradient
    5. Record the loss for that epoch
    """
    
    def __init__(self, learning_rate=0.01, n_epochs=1000):
        self.learning_rate = learning_rate  # α: step size
        self.n_epochs = n_epochs            # How many passes over data
        self.m = None   # Slope (coefficient for X)
        self.b = None   # Intercept (bias term)
        self.loss_history = []  # Track MSE each epoch
        self.m_history = []     # Track slope over time
        self.b_history = []     # Track intercept over time
    
    def _compute_mse(self, X, y):
        """MSE = (1/n) * sum((y_actual - y_predicted)^2)"""
        y_pred = self.m * X + self.b
        return np.mean((y - y_pred) ** 2)
    
    def fit(self, X, y):
        """
        TRAINING: Run gradient descent to find optimal m and b.
        
        Key Math:
        ---------
        Error      = y - (m*x + b)
        dJ/dm      = (-2/n) * sum(X * error)   [gradient w.r.t slope]
        dJ/db      = (-2/n) * sum(error)        [gradient w.r.t intercept]
        
        Note: (-2/n) factor comes from differentiating MSE.
        In practice, the 2 is often absorbed into learning_rate.
        """
        n = len(X)  # Number of training samples
        
        # 🚀 INITIALIZATION: Start from small random values
        # (Could also start from 0, but small random helps in neural networks)
        self.m = np.random.randn()  # Random slope
        self.b = np.random.randn()  # Random intercept
        
        print(f"Starting values → m={self.m:.4f}, b={self.b:.4f}")
        print(f"Target values  → m={true_slope}, b={true_intercept}")
        print("-" * 50)
        
        for epoch in range(self.n_epochs):
            
            # STEP 1: Compute predictions with CURRENT parameters
            # y_hat = m * X + b  (vectorized — applies to all samples at once)
            y_pred = self.m * X + self.b
            
            # STEP 2: Compute the error (residuals)
            error = y - y_pred  # Shape: (n,) — one error per sample
            
            # STEP 3: Compute gradients using ALL n samples
            # ∂J/∂m = (-2/n) * sum(X * error)
            # ∂J/∂b = (-2/n) * sum(error)
            grad_m = (-2 / n) * np.sum(X * error)  # Gradient for slope
            grad_b = (-2 / n) * np.sum(error)       # Gradient for intercept
            
            # STEP 4: UPDATE parameters (move opposite to gradient)
            self.m = self.m - self.learning_rate * grad_m
            self.b = self.b - self.learning_rate * grad_b
            
            # STEP 5: Record history for visualization later
            loss = self._compute_mse(X, y)
            self.loss_history.append(loss)
            self.m_history.append(self.m)
            self.b_history.append(self.b)
            
            # Print progress at key epochs
            if epoch % 200 == 0 or epoch == self.n_epochs - 1:
                print(f"Epoch {epoch:4d}: Loss={loss:.4f} | m={self.m:.4f} | b={self.b:.4f}")
        
        print("\n✅ Training complete!")
        print(f"Final m={self.m:.4f} (true={true_slope})")
        print(f"Final b={self.b:.4f} (true={true_intercept})")
        return self
    
    def predict(self, X):
        """Apply learned parameters to new data"""
        return self.m * X + self.b
    
    def score(self, X, y):
        """R² score — how well does our line fit?"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)


# 🏃 Run Batch Gradient Descent
bgd = BatchGradientDescent(learning_rate=0.01, n_epochs=1000)
bgd.fit(X_train, y_train)

In [ ]:
# ============================================================
#  📈 VISUALIZE: Loss Curve + Regression Line
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Loss Curve ---
# A decreasing loss curve = model is LEARNING
# The curve should flatten out = convergence (found minimum)
axes[0].plot(bgd.loss_history, color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('📉 Loss Curve (Batch GD)\nSmoothly decreasing = good sign!')
axes[0].set_yscale('log')  # Log scale to see early progress better

# --- Plot 2: Parameter Evolution ---
# Watch how m and b converge toward true values
axes[1].plot(bgd.m_history, label=f'm (slope) → {bgd.m:.3f}', color='blue')
axes[1].plot(bgd.b_history, label=f'b (intercept) → {bgd.b:.3f}', color='red')
axes[1].axhline(y=true_slope, color='blue', linestyle='--', alpha=0.5, label=f'True m={true_slope}')
axes[1].axhline(y=true_intercept, color='red', linestyle='--', alpha=0.5, label=f'True b={true_intercept}')
axes[1].set_xlabel('Epoch')
axes[1].set_title('🎯 Parameter Convergence\nm and b approaching true values')
axes[1].legend()

# --- Plot 3: Regression Line ---
axes[2].scatter(X_test, y_test, alpha=0.6, color='orange', label='Test data')
x_line = np.linspace(X_test.min(), X_test.max(), 100)
axes[2].plot(x_line, bgd.predict(x_line), 'b-', linewidth=2.5, label=f'BGD line (R²={bgd.score(X_test, y_test):.3f})')
axes[2].set_xlabel('CGPA')
axes[2].set_ylabel('Package (LPA)')
axes[2].set_title('🏆 Final Regression Line')
axes[2].legend()

plt.suptitle('Batch Gradient Descent Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚡ Part 4: Stochastic Gradient Descent (SGD)

### What is SGD?
- Uses **ONE random sample** at a time to compute the gradient
- Extremely fast updates (one update per sample)
- Very **noisy/erratic** path — high variance in updates
- Can escape local minima (noise helps!)
- Used in most **neural network** training

```
for each epoch:
    shuffle the data
    for each single sample:
        gradient = compute using JUST THIS ONE sample
        update parameters immediately
```

### Key Difference from Batch GD:
| Aspect | Batch GD | SGD |
|--------|----------|-----|
| Gradient computed on | All n samples | 1 sample |
| Updates per epoch | 1 | n |
| Stability | Very smooth | Very noisy |
| Speed per epoch | Slow | Fast |
| Memory | High | Low |


In [ ]:
# ============================================================
#  ⚡ STOCHASTIC GRADIENT DESCENT — Implementation from Scratch
# ============================================================

class StochasticGradientDescent:
    """
    Implements Linear Regression using STOCHASTIC Gradient Descent.
    
    KEY DIFFERENCE from Batch GD:
    - In Batch GD: gradient = average over ALL samples → 1 update per epoch
    - In SGD:      gradient = computed for 1 random sample → n updates per epoch
    
    This makes SGD:
    ✅ Much faster per epoch (especially for large datasets)
    ✅ Can escape local minima (due to noise)
    ❌ Loss curve is very noisy (zigzag pattern)
    ❌ May never perfectly converge (keeps bouncing around minimum)
    """
    
    def __init__(self, learning_rate=0.01, n_epochs=100):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.m = None
        self.b = None
        self.loss_history = []   # Loss after each epoch (not each sample)
        self.m_history = []
        self.b_history = []
    
    def fit(self, X, y):
        n = len(X)
        self.m = np.random.randn()
        self.b = np.random.randn()
        
        print(f"Starting: m={self.m:.4f}, b={self.b:.4f}")
        
        for epoch in range(self.n_epochs):
            
            # 🔀 SHUFFLE data at each epoch
            # Important! Without shuffling, model sees same order every time
            # which can cause systematic bias in updates
            indices = np.random.permutation(n)  # Random ordering of [0, 1, ..., n-1]
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # 🔁 Loop through each INDIVIDUAL sample
            for i in range(n):
                xi = X_shuffled[i]   # Single feature value (scalar)
                yi = y_shuffled[i]   # Single target value (scalar)
                
                # Prediction for this one sample
                y_pred_i = self.m * xi + self.b
                
                # Error for this one sample
                error_i = yi - y_pred_i
                
                # Gradient computed from JUST THIS ONE POINT
                # (No division by n because it's just 1 sample)
                grad_m = -2 * xi * error_i  # Gradient w.r.t. slope
                grad_b = -2 * error_i       # Gradient w.r.t. intercept
                
                # Immediate parameter update after each sample
                # This is what makes it "stochastic" — frequent, noisy updates
                self.m = self.m - self.learning_rate * grad_m
                self.b = self.b - self.learning_rate * grad_b
            
            # Record loss at end of each epoch (after seeing all samples)
            y_pred_all = self.m * X + self.b
            epoch_loss = np.mean((y - y_pred_all) ** 2)
            self.loss_history.append(epoch_loss)
            self.m_history.append(self.m)
            self.b_history.append(self.b)
            
            if epoch % 20 == 0:
                print(f"Epoch {epoch:3d}: Loss={epoch_loss:.4f} | m={self.m:.4f} | b={self.b:.4f}")
        
        print(f"\n✅ Done! Final m={self.m:.4f}, b={self.b:.4f}")
        return self
    
    def predict(self, X):
        return self.m * X + self.b
    
    def score(self, X, y):
        return r2_score(y, self.predict(X))


# 🏃 Run SGD
sgd = StochasticGradientDescent(learning_rate=0.001, n_epochs=100)
sgd.fit(X_train, y_train)

---
## 🍜 Part 5: Mini-Batch Gradient Descent

### What is Mini-Batch GD?
- The **BEST of both worlds** between Batch GD and SGD
- Uses a **small batch** of samples (e.g., 32, 64, 128) per update
- Less noisy than SGD, faster than Batch GD
- **This is what's used in practice for Deep Learning!**

### Why Does Batch Size Matter?
| Batch Size | Behavior |
|------------|----------|
| = 1 | → Same as SGD (most noisy) |
| = n | → Same as Batch GD (smoothest) |
| 32–256 | → Sweet spot (industry default: 32 or 64) |

```
for each epoch:
    shuffle data
    split into mini-batches of size k
    for each mini-batch:
        gradient = compute using k samples
        update parameters
```

In [ ]:
# ============================================================
#  🍜 MINI-BATCH GRADIENT DESCENT — Implementation from Scratch
# ============================================================

class MiniBatchGradientDescent:
    """
    Implements Linear Regression using MINI-BATCH Gradient Descent.
    
    INTUITION:
    -  Batch GD → reads whole book before writing notes   (slow, accurate)
    -  SGD      → reads one sentence, writes notes, repeats (fast, noisy)
    -  Mini-Batch → reads a chapter, writes notes, repeats  (balanced!)
    
    In practice:
    - GPU computation is parallelized, so batch_size=32 is nearly as fast as batch_size=1
    - But the gradient estimate is much more stable with a small batch
    - This is the default in PyTorch, TensorFlow, Keras, etc.
    """
    
    def __init__(self, learning_rate=0.01, n_epochs=100, batch_size=32):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size  # How many samples per gradient update
        self.m = None
        self.b = None
        self.loss_history = []
        self.m_history = []
        self.b_history = []
    
    def fit(self, X, y):
        n = len(X)
        self.m = np.random.randn()
        self.b = np.random.randn()
        
        # How many batches per epoch?
        # If n=160, batch_size=32 → 5 batches per epoch
        n_batches = int(np.ceil(n / self.batch_size))
        print(f"Samples: {n}, Batch size: {self.batch_size}, Batches/epoch: {n_batches}")
        print(f"Starting: m={self.m:.4f}, b={self.b:.4f}")
        
        for epoch in range(self.n_epochs):
            
            # 🔀 Shuffle data before splitting into batches
            # Ensures different batches each epoch → better generalization
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # 📦 Process each mini-batch
            for batch_idx in range(n_batches):
                
                # Slice the mini-batch from the shuffled data
                start = batch_idx * self.batch_size
                end   = start + self.batch_size  # np handles if end > n
                
                X_batch = X_shuffled[start:end]  # Shape: (batch_size,)
                y_batch = y_shuffled[start:end]  # Shape: (batch_size,)
                
                batch_n = len(X_batch)  # Actual batch size (last batch may be smaller)
                
                # Prediction for this mini-batch
                y_pred_batch = self.m * X_batch + self.b
                
                # Error for this mini-batch
                error_batch = y_batch - y_pred_batch
                
                # Gradient computed from BATCH_SIZE samples
                # (More stable than SGD, faster than Batch GD)
                grad_m = (-2 / batch_n) * np.sum(X_batch * error_batch)
                grad_b = (-2 / batch_n) * np.sum(error_batch)
                
                # Update parameters after each mini-batch
                self.m = self.m - self.learning_rate * grad_m
                self.b = self.b - self.learning_rate * grad_b
            
            # Record epoch-level loss (after all batches processed)
            y_pred_all = self.m * X + self.b
            epoch_loss = np.mean((y - y_pred_all) ** 2)
            self.loss_history.append(epoch_loss)
            self.m_history.append(self.m)
            self.b_history.append(self.b)
            
            if epoch % 20 == 0:
                print(f"Epoch {epoch:3d}: Loss={epoch_loss:.4f} | m={self.m:.4f} | b={self.b:.4f}")
        
        print(f"\n✅ Done! Final m={self.m:.4f}, b={self.b:.4f}")
        return self
    
    def predict(self, X):
        return self.m * X + self.b
    
    def score(self, X, y):
        return r2_score(y, self.predict(X))


# 🏃 Run Mini-Batch GD with batch_size=32
mbgd = MiniBatchGradientDescent(learning_rate=0.01, n_epochs=100, batch_size=32)
mbgd.fit(X_train, y_train)

In [ ]:
# ============================================================
#  📊 SIDE-BY-SIDE COMPARISON: All Three Methods
# ============================================================

# Re-run all three with same # of epochs for fair comparison
bgd2  = BatchGradientDescent(learning_rate=0.01, n_epochs=100)
bgd2.fit(X_train, y_train)

sgd2  = StochasticGradientDescent(learning_rate=0.001, n_epochs=100)
sgd2.fit(X_train, y_train)

mbgd2 = MiniBatchGradientDescent(learning_rate=0.01, n_epochs=100, batch_size=32)
mbgd2.fit(X_train, y_train)

In [ ]:
# ============================================================
#  📈 COMPARISON PLOT: Loss Curves of All Three Methods
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = [
    (bgd2,  '🔵 Batch GD',      'blue',   'Smooth but slow per update'),
    (sgd2,  '⚡ Stochastic GD', 'red',    'Noisy but many updates/epoch'),
    (mbgd2, '🍜 Mini-Batch GD', 'green',  'Best balance — industry standard'),
]

for ax, (model, name, color, note) in zip(axes, methods):
    ax.plot(model.loss_history, color=color, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE Loss', fontsize=11)
    ax.set_title(f'{name}\n{note}', fontsize=11, fontweight='bold')
    
    # Annotate final loss
    final_loss = model.loss_history[-1]
    ax.annotate(f'Final: {final_loss:.3f}', 
                xy=(len(model.loss_history)-1, final_loss),
                xytext=(-60, 10), textcoords='offset points',
                fontsize=10, color=color)

plt.suptitle('Loss Curves: Batch vs SGD vs Mini-Batch Gradient Descent', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Performance summary
print("\n" + "="*65)
print(f"{'Method':<20} {'R² Score':<12} {'Final Loss':<15} {'m (slope)':<12} {'b (intercept)'}")
print("="*65)
for model, name, _, _ in methods:
    r2  = model.score(X_test, y_test)
    mse = model.loss_history[-1]
    print(f"{name:<20} {r2:<12.4f} {mse:<15.4f} {model.m:<12.4f} {model.b:.4f}")
print(f"{'True values':<20} {'—':<12} {'—':<15} {true_slope:<12} {true_intercept}")
print("="*65)

---
## 🎛️ Part 6: Learning Rate — The Most Critical Hyperparameter

The learning rate $\alpha$ controls **how big a step** we take in the direction of the gradient.

| Learning Rate | Behavior | Result |
|---------------|----------|--------|
| Too **small** | Tiny steps | Very slow convergence (takes forever) |
| **Just right** | Appropriate steps | Converges smoothly and efficiently |
| Too **large** | Huge steps | Overshoots minimum, may diverge |

Common starting values: `0.001`, `0.01`, `0.1` (try on log scale)

In [ ]:
# ============================================================
#  🎛️ LEARNING RATE SENSITIVITY ANALYSIS
# ============================================================

learning_rates = [0.0001, 0.001, 0.01, 0.1, 0.5]
colors = ['purple', 'blue', 'green', 'orange', 'red']
labels = ['Too small (α=0.0001)', 'Small (α=0.001)', 
          '✅ Good (α=0.01)', 'Aggressive (α=0.1)', 
          '❌ Too large (α=0.5)']

plt.figure(figsize=(12, 6))

for lr, color, label in zip(learning_rates, colors, labels):
    model = BatchGradientDescent(learning_rate=lr, n_epochs=200)
    # Suppress output for this section
    import io, sys
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    model.fit(X_train, y_train)
    sys.stdout = old_stdout
    
    # Clip extreme values for visibility
    losses = np.clip(model.loss_history, 0, 500)
    plt.plot(losses, color=color, label=label, linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Learning Rate Sensitivity\nHow α affects convergence speed & stability', fontsize=13)
plt.legend(loc='upper right')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

print("Key Takeaways:")
print("  • α too small  → converges but very slowly (still decreasing at epoch 200)")
print("  • α just right → converges quickly and smoothly")
print("  • α too large  → loss may increase or oscillate wildly")

---
## 🌄 Part 7: 3D Visualization of Loss Surface

The **loss landscape** is a surface in parameter space. Each point (m, b) has a corresponding loss value.

Gradient Descent is like a ball rolling downhill on this surface to find the lowest point (minimum loss).

In [ ]:
# ============================================================
#  🌄 3D LOSS SURFACE VISUALIZATION
# ============================================================

from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# Create a grid of (m, b) values
m_vals = np.linspace(0, 7, 80)      # Range of possible slopes
b_vals = np.linspace(-5, 10, 80)    # Range of possible intercepts
M, B = np.meshgrid(m_vals, b_vals)  # Create 2D grid

# Compute MSE loss for EACH (m, b) combination
# J(m, b) = (1/n) * sum((y - (m*x + b))^2)
Loss = np.zeros_like(M)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        y_pred = M[i, j] * X_train + B[i, j]  # Predictions for all train samples
        Loss[i, j] = np.mean((y_train - y_pred) ** 2)  # MSE

# Clip for visualization
Loss_clipped = np.clip(Loss, 0, 100)

fig = plt.figure(figsize=(16, 6))

# --- 3D Surface Plot ---
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(M, B, Loss_clipped, cmap=cm.viridis, alpha=0.8)

# Mark the minimum (optimal parameters)
ax1.scatter([bgd2.m], [bgd2.b], [bgd2.loss_history[-1]], 
            c='red', s=100, marker='*', zorder=10, label='GD solution')
ax1.scatter([true_slope], [true_intercept], 
            [np.mean((y_train - true_slope * X_train - true_intercept)**2)],
            c='gold', s=100, marker='D', zorder=10, label='True minimum')

ax1.set_xlabel('Slope (m)')
ax1.set_ylabel('Intercept (b)')
ax1.set_zlabel('MSE Loss')
ax1.set_title('3D Loss Surface\nGradient Descent finds the bowl\'s bottom!')
ax1.legend()
plt.colorbar(surf, ax=ax1, shrink=0.5)

# --- 2D Contour Plot (Top-down view) ---
ax2 = fig.add_subplot(122)
contour = ax2.contourf(M, B, Loss_clipped, levels=50, cmap='viridis')
plt.colorbar(contour, ax=ax2)

# Plot the path taken by Batch GD
# (Darker = later epoch, should spiral toward minimum)
bgd_path_m = bgd2.m_history
bgd_path_b = bgd2.b_history
ax2.plot(bgd_path_m, bgd_path_b, 'r.-', markersize=3, linewidth=1, 
         label='Batch GD path', alpha=0.7)
ax2.scatter([bgd_path_m[0]], [bgd_path_b[0]], c='white', s=100, 
            marker='o', zorder=5, label='Start')
ax2.scatter([bgd_path_m[-1]], [bgd_path_b[-1]], c='red', s=100, 
            marker='*', zorder=5, label='End (GD)')
ax2.scatter([true_slope], [true_intercept], c='gold', s=150, 
            marker='D', zorder=5, label='True values')

ax2.set_xlabel('Slope (m)')
ax2.set_ylabel('Intercept (b)')
ax2.set_title('Contour Map (Top-down view)\nRed path = GD trajectory')
ax2.legend()

plt.suptitle('Loss Landscape: The Bowl Gradient Descent Navigates', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 Part 8: Feature Scaling — Why It Matters for GD

When features have different scales, the loss surface becomes an **elongated ellipse** instead of a circle. This causes GD to take a zigzag path and converge slowly.

**Solution: Standardize features (mean=0, std=1) before running GD!**

```
x_scaled = (x - mean) / std
```

In [ ]:
# ============================================================
#  🔧 FEATURE SCALING DEMO — Why Scaling Helps GD
# ============================================================
from sklearn.datasets import make_regression

# Create a dataset with TWO features at very different scales
np.random.seed(42)
n = 200

X1_raw = np.random.normal(50, 10, n)      # Feature 1: ~50, range [20, 80]
X2_raw = np.random.normal(0.5, 0.1, n)   # Feature 2: ~0.5, range [0.2, 0.8]
y_multi = 3 * X1_raw + 500 * X2_raw + np.random.normal(0, 5, n)

X_multi = np.column_stack([X1_raw, X2_raw])

# Without scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_multi)

print("BEFORE SCALING:")
print(f"  Feature 1: mean={X_multi[:,0].mean():.1f}, std={X_multi[:,0].std():.1f}")
print(f"  Feature 2: mean={X_multi[:,1].mean():.3f}, std={X_multi[:,1].std():.3f}")
print()
print("AFTER SCALING (StandardScaler):")
print(f"  Feature 1: mean={X_scaled[:,0].mean():.4f}, std={X_scaled[:,0].std():.4f}")
print(f"  Feature 2: mean={X_scaled[:,1].mean():.4f}, std={X_scaled[:,1].std():.4f}")
print()
print("✅ Both features now have mean ≈ 0 and std ≈ 1")
print("✅ GD will converge much faster and more reliably!")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_multi[:,0], X_multi[:,1], alpha=0.5, color='red')
axes[0].set_title('Before Scaling\n(Features on very different scales)')
axes[0].set_xlabel('Feature 1 (scale: ~50)')
axes[0].set_ylabel('Feature 2 (scale: ~0.5)')

axes[1].scatter(X_scaled[:,0], X_scaled[:,1], alpha=0.5, color='green')
axes[1].set_title('After StandardScaler\n(Both features on same scale)')
axes[1].set_xlabel('Feature 1 (scaled)')
axes[1].set_ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()

---
## 🆚 Part 9: Compare with Sklearn's Built-in Models

In [ ]:
# ============================================================
#  🆚 COMPARE OUR SCRATCH IMPLEMENTATION vs SKLEARN
# ============================================================

# Sklearn's LinearRegression uses Normal Equation (exact analytical solution)
sklearn_lr = LinearRegression()
sklearn_lr.fit(X_train.reshape(-1, 1), y_train)
sklearn_r2 = r2_score(y_test, sklearn_lr.predict(X_test.reshape(-1, 1)))

# Sklearn's SGD Regressor (their implementation of SGD)
sklearn_sgd = SGDRegressor(learning_rate='constant', eta0=0.001, max_iter=200, random_state=42)
sklearn_sgd.fit(X_train.reshape(-1, 1), y_train)
sklearn_sgd_r2 = r2_score(y_test, sklearn_sgd.predict(X_test.reshape(-1, 1)))

print("📊 Model Comparison Summary")
print("="*60)
print(f"{'Model':<30} {'R² Score':<12} {'Slope':<10} {'Intercept'}")
print("-"*60)
print(f"{'True values':<30} {'N/A':<12} {true_slope:<10} {true_intercept}")
print(f"{'Sklearn Linear Reg (Normal Eq)':<30} {sklearn_lr_r2:.4f}       {sklearn_lr.coef_[0]:.4f}     {sklearn_lr.intercept_:.4f}")
print(f"{'Our Batch GD':<30} {bgd2.score(X_test, y_test):.4f}       {bgd2.m:.4f}     {bgd2.b:.4f}")
print(f"{'Our SGD':<30} {sgd2.score(X_test, y_test):.4f}       {sgd2.m:.4f}     {sgd2.b:.4f}")
print(f"{'Our Mini-Batch GD':<30} {mbgd2.score(X_test, y_test):.4f}       {mbgd2.m:.4f}     {mbgd2.b:.4f}")
print(f"{'Sklearn SGD Regressor':<30} {sklearn_sgd_r2:.4f}       {sklearn_sgd.coef_[0]:.4f}     {sklearn_sgd.intercept_[0]:.4f}")
print("="*60)

# Oops — need to fix the variable name
sklearn_lr_r2 = sklearn_r2

In [ ]:
# Fix and display cleanly
sklearn_lr_r2 = r2_score(y_test, sklearn_lr.predict(X_test.reshape(-1, 1)))

print("📊 FINAL Model Comparison")
print("="*60)
print(f"{'Model':<30} {'R² Score':>10}")
print("-"*60)
print(f"{'Sklearn Linear Reg':<30} {sklearn_lr_r2:>10.4f}  ← Gold standard (exact)")
print(f"{'Our Batch GD':<30} {bgd2.score(X_test, y_test):>10.4f}")
print(f"{'Our Mini-Batch GD':<30} {mbgd2.score(X_test, y_test):>10.4f}")
print(f"{'Our SGD':<30} {sgd2.score(X_test, y_test):>10.4f}")
print(f"{'Sklearn SGD':<30} {sklearn_sgd_r2:>10.4f}")
print("="*60)
print("\n✅ Our implementations match Sklearn's results!")

---
## 💡 Part 10: Common Pitfalls & Best Practices

### ⚠️ Problems in Gradient Descent

| Problem | Symptom | Fix |
|---------|---------|-----|
| Learning rate too high | Loss increases / oscillates | Reduce α |
| Learning rate too low | Converges very slowly | Increase α |
| Features not scaled | Slow / zigzag convergence | Use StandardScaler |
| Local minima | Stops before true minimum | Use SGD noise, momentum, Adam |
| Saddle points | Flat areas that aren't minimum | Momentum-based optimizers |
| Vanishing gradient | Gradients become zero | Better activation functions, BatchNorm |

### ✅ Best Practices
1. **Always scale features** before using GD
2. **Start with α=0.01**, then tune
3. **Plot the loss curve** — should decrease monotonically (Batch) or trend down (SGD)
4. **Use Mini-Batch GD** for deep learning (batch_size=32 is a great default)
5. **Consider adaptive optimizers** (Adam, RMSprop) for complex models

### 🧮 Gradient Descent Variants Beyond the Basics

| Optimizer | Idea | Use Case |
|-----------|------|----------|
| **SGD + Momentum** | Adds velocity term to avoid oscillation | CNNs |
| **RMSprop** | Adapts learning rate per parameter | RNNs |
| **Adam** | Combines Momentum + RMSprop | Default for most Deep Learning |
| **AdaGrad** | Larger LR for rare features | Sparse data, NLP |

In [ ]:
# ============================================================
#  🏁 SUMMARY TABLE — Quick Reference
# ============================================================

summary = {
    'Method'          : ['Batch GD', 'Stochastic GD', 'Mini-Batch GD'],
    'Samples/Update'  : ['All n', '1', 'k (e.g. 32, 64)'],
    'Updates/Epoch'   : ['1', 'n', 'n/k'],
    'Convergence'     : ['Smooth', 'Noisy/erratic', 'Moderately smooth'],
    'Memory Usage'    : ['High', 'Very low', 'Low'],
    'Speed (large n)' : ['Slow', 'Fast', 'Fast'],
    'Used In'         : ['Small datasets', 'Online learning', 'Deep Learning ✅'],
    'sklearn class'   : ['N/A (use Normal Eq)', 'SGDRegressor(batch=1)', 'SGDRegressor'],
}

df_summary = pd.DataFrame(summary)
df_summary = df_summary.set_index('Method')
print("\n📋 GRADIENT DESCENT COMPARISON TABLE")
print("="*80)
print(df_summary.T.to_string())
print("="*80)

print("\n🎯 RULE OF THUMB:")
print("  Small dataset (< 10k)  → Batch GD or Mini-Batch")
print("  Large dataset           → Mini-Batch GD (batch_size = 32 or 64)")
print("  Streaming/online data  → SGD")
print("  Deep Neural Networks   → Mini-Batch + Adam optimizer")

---
## 🎓 Summary

### What We Learned:

**Day 51 — Gradient Descent:**
- GD is an optimization algorithm that minimizes a loss function iteratively
- The update rule: `θ = θ - α * ∂J/∂θ`
- Learning rate α is the most critical hyperparameter
- We built Batch GD from scratch and verified it matches sklearn

**Day 52 — Types of Gradient Descent:**
- **Batch GD** → all data, smooth convergence, slow for large datasets
- **SGD** → 1 sample, fast but noisy, can escape local minima
- **Mini-Batch GD** → k samples, best trade-off, industry standard

**Key Extras:**
- Feature scaling is **mandatory** for GD to work properly
- The loss surface is a 3D bowl (for convex problems like Linear Regression)
- Modern deep learning uses Adam optimizer (extension of Mini-Batch GD)

---
### 📚 Next Steps:
- Day 53: Polynomial Regression (GD with non-linear features)
- Learn about **Adam, RMSprop, Momentum** optimizers
- Explore **learning rate scheduling** (decreasing α over time)
- Apply GD to **Logistic Regression** (Day 58)

---
*Enhanced from CampusX 100 Days of Machine Learning — Day 51 & 52*